#  Laboratório 10: O Pipeline Definitivo
## RAG + QLoRA + Otimização de Inferência em GPU

> **Declaração Obrigatória:** Partes deste laboratório foram geradas/complementadas com IA, revisadas e validadas por Bruno

---

###  ANTES DE COMEÇAR — Ative a GPU!
```
Menu → Ambiente de execução → Alterar tipo de hardware → GPU → Salvar
```

###  Nota sobre Limitação de Hardware (T4 Colab Free)

Este laboratório foi desenvolvido no **Google Colab Free (GPU T4, 16 GB VRAM)**.

| Limitação | Causa | Solução aplicada |
|---|---|---|
| FlashAttention-2 indisponível | T4 = Turing sm=75; FA2 exige Ampere sm>=80 | try/except com fallback automático |
| Contexto reduzido para 2.000 tokens | 12k tokens sem cache causa OOM/timeout no T4 | TOKENS_ALVO = 2_000 |

> O código está **correto e completo**. As métricas são reais para o T4.
> Em A100/L4 com FA2 ativo e 12k+ tokens os ganhos seriam ainda maiores.

### Pipeline
```
[RAG simulado] --> [Llama QLoRA 4-bit] --> [Resumo Clínico]
  2.000 tokens       KV Cache + FA2*        100 tokens
  (*fallback no T4)
```


In [ ]:

import subprocess

resultado = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                            "--format=csv,noheader"], capture_output=True, text=True)

if resultado.returncode == 0:
    nome_gpu, vram_total = resultado.stdout.strip().split(",")
    print(f"✅ GPU detectada : {nome_gpu.strip()}")
    print(f"   VRAM total    : {vram_total.strip()}")

    gpus_com_fa2 = ["A100", "A10", "L4", "H100", "RTX 30", "RTX 40", "A6000"]
    suporta_fa2 = any(g in nome_gpu for g in gpus_com_fa2)
    if suporta_fa2:
        print("   FlashAttention-2: ✅ suportada nesta GPU!")
    else:
        print("   FlashAttention-2: ⚠️ não suportada (T4=Turing) → usará fallback automático")
        print("   O laboratório funciona normalmente mesmo sem FA2.")
else:
    print("❌ Nenhuma GPU encontrada!")
    print("   Vá em: Ambiente de execução → Alterar tipo de hardware → GPU")


##   Instalação das Dependências

  **Após executar esta célula, clique em "Reiniciar sessão" quando solicitado!**
 Depois reinicie a partir da Célula 2 — não execute a instalação novamente.

In [ ]:


print("[1/4] Instalando bitsandbytes (quantização QLoRA)...")
!pip install -q bitsandbytes>=0.43.0

print("[2/4] Instalando transformers + accelerate...")
!pip install -q transformers>=4.40.0 accelerate>=0.30.0

print("[3/4] Instalando psutil + matplotlib...")
!pip install -q psutil matplotlib

print("[4/4] Tentando instalar FlashAttention-2 (pode demorar ~5 min)...")
print("      Se falhar no T4, o laboratório usa fallback automático.")
try:
    import subprocess
    proc = subprocess.run(
        ["pip", "install", "-q", "flash-attn", "--no-build-isolation"],
        capture_output=True, text=True, timeout=600
    )
    if proc.returncode == 0:
        print("       flash-attn instalado com sucesso!")
    else:
        print("       flash-attn não instalou — FA2 usará fallback no Passo 4b")
except Exception as e:
    print(f"       Timeout/erro no flash-attn: {e}")

print("\n" + "═"*55)
print("   Instalação concluída!")
print("   Clique em 'Reiniciar sessão' e depois continue")
print("     a partir da Célula 2 (não rode esta célula de novo).")
print("═"*55)


## Imports e Utilitários
 **Comece aqui após reiniciar a sessão**

In [ ]:
import time
import textwrap
import warnings
warnings.filterwarnings("ignore")

import torch
import psutil
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig



def gpu_disponivel() -> bool:
    """Retorna True se houver GPU CUDA disponível."""
    return torch.cuda.is_available()

def resetar_stats_vram() -> None:
    """Zera o contador de pico de VRAM para medição limpa."""
    if gpu_disponivel():
        torch.cuda.reset_peak_memory_stats()

def vram_atual_mb() -> float:
    """VRAM atualmente alocada em MB."""
    return torch.cuda.memory_allocated() / 1024**2 if gpu_disponivel() else 0.0

def pico_vram_mb() -> float:
    """Pico de VRAM desde o último reset, em MB."""
    return torch.cuda.max_memory_allocated() / 1024**2 if gpu_disponivel() else 0.0

def vram_livre_mb() -> float:
    """VRAM livre disponível em MB."""
    if gpu_disponivel():
        livre, total = torch.cuda.mem_get_info()
        return livre / 1024**2
    return 0.0


DEVICE = "cuda" if gpu_disponivel() else "cpu"

print("═" * 55)
print("  LAB 10 — Diagnóstico do Ambiente Colab")
print("═" * 55)
if gpu_disponivel():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU    : {props.name}")
    print(f"  VRAM   : {props.total_memory / 1024**3:.1f} GB total")
    print(f"  Livre  : {vram_livre_mb():.0f} MB disponíveis agora")
else:
    print("   GPU não encontrada — ative em Ambiente de execução!")
print(f"  PyTorch: {torch.__version__}")
print(f"  Device : {DEVICE}")
print("═" * 55)


##  Carregar Modelo com QLoRA 4-bit

### Por que quantizar?

```
Float16 (sem quant): 2 bytes/param  → TinyLlama 1.1B = ~2.200 MB  
QLoRA NF4 (4-bit) : 0,5 bytes/param → TinyLlama 1.1B =   ~600 MB  
```

O cálculo ainda ocorre em FP16 após dequantização — qualidade preservada!

In [ ]:

ID_MODELO = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Modelo  : {ID_MODELO}")
print(f"Device  : {DEVICE}")
print(f"VRAM livre antes : {vram_livre_mb():.0f} MB\n")

config_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Carregando tokenizador...")
tokenizador = AutoTokenizer.from_pretrained(ID_MODELO)
tokenizador.pad_token = tokenizador.eos_token

print("Carregando modelo com QLoRA 4-bit...")
resetar_stats_vram()
vram_antes = vram_atual_mb()

modelo = AutoModelForCausalLM.from_pretrained(
    ID_MODELO,
    quantization_config=config_bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
modelo.eval()

vram_depois   = vram_atual_mb()
pegada_mb     = vram_depois - vram_antes

print("\n" + "═" * 50)
print(" 📊 MÉTRICA — Passo 1: Carga do Modelo")
print("═" * 50)
print(f"  VRAM antes : {vram_antes:>8.1f} MB")
print(f"  VRAM depois: {vram_depois:>8.1f} MB")
print(f"  Pegada     : {pegada_mb:>8.1f} MB  ← ~4× menos que FP16!")
print(f"  VRAM livre : {vram_livre_mb():>8.0f} MB restantes")
print("═" * 50)


##   Simular o Contexto RAG Massivo (~12.000 tokens)

Simulamos os **5 capítulos de manuais médicos** que seriam recuperados pelo banco vetorial RAG,
cobrindo cardiologia, farmacologia, IAM, doença renal e documentação clínica.

In [ ]:


CAPITULOS = [

    """CAPÍTULO 1: Fisiologia Cardiovascular e Hemodinâmica
O sistema cardiovascular é responsável pela circulação contínua do sangue pelo organismo,
transportando oxigênio e nutrientes aos tecidos enquanto remove dióxido de carbono e produtos
metabólicos. O coração, uma bomba muscular, gera a pressão necessária para impulsionar o sangue
por dois circuitos distintos: a circulação pulmonar, que oxigena o sangue nos pulmões, e a
circulação sistêmica, que distribui o sangue oxigenado aos tecidos periféricos.
O débito cardíaco (DC), definido como o volume de sangue ejetado por minuto, é o produto da
frequência cardíaca (FC) pelo volume sistólico (VS): DC = FC × VS. Em adultos em repouso, o
débito cardíaco normal varia de 4 a 8 litros por minuto. O volume sistólico é influenciado pela
pré-carga (volume diastólico final), pós-carga (resistência arterial) e contratilidade miocárdica.
A pré-carga é governada pelo retorno venoso e descrita pelo mecanismo de Frank-Starling:
dentro dos limites fisiológicos, maior enchimento ventricular resulta em contração mais vigorosa.
A pós-carga depende da resistência vascular sistêmica (RVS), complacência arterial e viscosidade
sanguínea. A hipertensão aumenta a pós-carga, forçando o ventrículo a gerar maior estresse
parietal, levando a remodelação hipertrófica e disfunção diastólica ao longo do tempo.
A pressão arterial média (PAM) é estimada como: PAM = PAD + 1/3 (PAS − PAD).""",

    """CAPÍTULO 2: Manejo Farmacológico da Hipertensão Arterial
A hipertensão arterial sistêmica afeta aproximadamente 1,28 bilhão de adultos no mundo e é
o principal fator de risco modificável para infarto, AVC, insuficiência cardíaca e doença renal.
A farmacoterapia é iniciada quando intervenções no estilo de vida (restrição de sódio, perda
de peso, exercício aeróbico, dieta DASH) não alcançam a meta abaixo de 130/80 mmHg.
As classes de primeira linha incluem: diuréticos tiazídicos, bloqueadores dos canais de cálcio
(BCC), inibidores da ECA (IECA) e bloqueadores dos receptores de angiotensina (BRA).
Diuréticos tiazídicos (hidroclorotiazida, clortalidona) reduzem o volume plasmático agudamente;
efeitos crônicos são mediados por vasodilatação. BCCs diidropiridínicos (anlodipino, nifedipino)
atuam no músculo liso vascular; não-diidropiridínicos (diltiazem, verapamil) também reduzem
frequência cardíaca e contratilidade. IECAs bloqueiam a conversão de angiotensina I em
angiotensina II e impedem a degradação da bradicinina (causando tosse seca característica).
BRAs bloqueiam seletivamente o receptor AT1 com eficácia similar sem o efeito de tosse.
Betabloqueadores são preferidos na hipertensão complicada por DAC ou IC com FE reduzida.""",

    """CAPÍTULO 3: Infarto Agudo do Miocárdio — Diagnóstico e Intervenção
O infarto agudo do miocárdio (IAM) resulta da cessação abrupta do fluxo coronariano, levando
à isquemia e necrose irreversível de cardiomiócitos se a reperfusão não for restaurada em 2–4h.
A definição universal classifica o IAM em cinco tipos: Tipo 1 (ruptura de placa aterosclerótica),
Tipo 2 (desequilíbrio oferta-demanda de oxigênio), Tipo 3 (morte cardíaca com IAM presumido)
e Tipos 4–5 (relacionados a procedimentos). O IAMCSST é diagnosticado quando há elevação de
ST ≥ 1 mm em duas derivações periféricas contíguas ou ≥ 2 mm em derivações precordiais,
indicando oclusão coronariana total. O manejo segue o princípio "tempo é músculo": meta de
primeiro contato médico ao balão ≤ 90 minutos. A ICP primária é a estratégia de reperfusão
preferencial. Terapia antiplaquetária com AAS (325 mg de ataque) mais inibidor de P2Y12
(ticagrelor 180 mg ou prasugrel 60 mg) é administrada imediatamente. Anticoagulação com
heparina não fracionada (HNF) ou bivalirudina é padrão durante a ICP. Prevenção secundária
inclui dupla antiagregação por 12 meses, estatina de alta intensidade, IECA/BRA e
betabloqueador por ao menos 3 anos em pacientes com FE reduzida.""",

    """CAPÍTULO 4: Doença Renal Crônica — Estadiamento e Síndrome Cardiorrenal
A doença renal crônica (DRC) é definida pela presença de lesão renal ou TFG inferior a
60 mL/min/1,73 m² por mais de três meses. A classificação KDIGO estadia de G1 (TFG ≥ 90)
a G5 (TFG < 15, dialise). Categorias de albuminúria (A1–A3) estratificam o risco adicional.
A DRC está fortemente ligada à doença cardiovascular de forma bidirecional — a síndrome
cardiorrenal (SCR) engloba cinco subtipos descrevendo interações agudas e crônicas entre
disfunção cardíaca e renal. Mecanismos incluem ativação neuro-hormonal (SRAA, SNS), retenção
de fluidos com aumento da pré-carga, toxinas urêmicas comprometendo função endotelial e
anemia da doença crônica reduzindo transporte de oxigênio. A TFGe é calculada pela equação
CKD-EPI 2021 para creatinina (sem raça como variável). IECAs e BRAs são o pilar da terapia
nefroprotetora na DRC com proteinúria. Os iSGLT2 (dapagliflozina, empagliflozina) demonstraram
efeitos nefroprotetores independentes do controle glicêmico, reduzindo progressão da DRC em
39% nos ensaios DAPA-CKD e EMPA-KIDNEY.""",

    """CAPÍTULO 5: Documentação Clínica e Laudos Médicos Estruturados
A documentação clínica de alta qualidade é o alicerce de uma assistência segura, eficaz e
juridicamente defensável. O prontuário médico comunica informações críticas entre profissionais,
apoia decisões clínicas, viabiliza melhoria da qualidade e fornece evidências para reembolso
e processos medicolegais. O formato SOAP (Subjetivo, Objetivo, Avaliação, Plano) estrutura
consultas ambulatoriais. Documentação hospitalar inclui HCEA de admissão, evoluções diárias,
notas de procedimentos, relatórios operatórios e resumos de alta. Resumos de alta devem conter:
diagnóstico principal, diagnósticos secundários, procedimentos realizados, achados significativos
(laboratório, imagem, patologia), narrativa da evolução, medicamentos na alta com doses e vias,
instruções de acompanhamento e resultados pendentes. Codificação pelo CID-10 requer
especificidade sobre lateralidade, cronicidade e etiologia. Codificação de procedimentos deve
refletir a complexidade da tomada de decisão médica conforme diretrizes atualizadas de E/M.""",
]


SUFIXO_TAREFA = (
    "\n\n---\n"
    "TAREFA CLÍNICA: Com base na documentação médica acima, gere um resumo clínico "
    "conciso de 500 palavras sobre o estado cardiovascular, função renal, "
    "manejo farmacológico e ações de acompanhamento recomendadas."
)


TOKENS_ALVO = 2_000

corpus_base = "\n\n".join(CAPITULOS)
corpus_expandido = corpus_base

print("Expandindo corpus até o alvo de tokens...")
while True:
    ids = tokenizador(corpus_expandido, return_tensors="pt").input_ids
    if ids.shape[1] >= TOKENS_ALVO:
        break
    corpus_expandido += "\n\n" + corpus_base


ids_recortados  = ids[:, :TOKENS_ALVO]
contexto_massivo = tokenizador.decode(ids_recortados[0], skip_special_tokens=True)

prompt_completo = contexto_massivo + SUFIXO_TAREFA

input_ids = tokenizador(
    prompt_completo,
    return_tensors="pt",
    truncation=True,
    max_length=2_500,
).input_ids.to(DEVICE)

n_tokens_entrada = input_ids.shape[1]

print("═" * 52)
print("  MÉTRICA — Passo 2: Contexto RAG Tokenizado")
print("═" * 52)
print(f"  Tokens no contexto  : {n_tokens_entrada:,} tokens")
print(f"  Palavras aprox.     : {len(prompt_completo.split()):,} palavras")
print(f"  Shape do tensor     : {list(input_ids.shape)}")
print("═" * 52)

##   Gargalo: Geração SEM KV Cache

Sem cache, cada novo token recalcula Q, K e V **para toda a sequência**:

```
Token 1 → atenção em [10000 tokens]   → O(n²)
Token 2 → atenção em [10001 tokens]   → recalcula TUDO de novo!
...      (100 vezes)
```

In [ ]:
NOVOS_TOKENS = 50


print("[Passo 3] Gerando SEM KV Cache (caminho lento)...")
print(f"          Entrada : {n_tokens_entrada:,} tokens | Gerando : {NOVOS_TOKENS} tokens")
print("          No T4 Free isso pode demorar 2-5 min. É normal!\n")
          print("          Observe: sem cache, cada token recalcula Q,K,V para toda a sequência.\n")

modelo.config.use_cache = False
resetar_stats_vram()

t_inicio = time.perf_counter()

with torch.no_grad():
    saida_sem_cache = modelo.generate(
        input_ids,
        max_new_tokens=NOVOS_TOKENS,
        do_sample=False,
        temperature=1.0,
        use_cache=False,
    )

t_sem_cache    = time.perf_counter() - t_inicio
vram_sem_cache = pico_vram_mb()


texto_gerado_sc = tokenizador.decode(
    saida_sem_cache[0, n_tokens_entrada:], skip_special_tokens=True
)

print("═" * 52)
print("  MÉTRICA — Passo 3: SEM KV Cache")
print("═" * 52)
print(f"  Tempo de geração   : {t_sem_cache:>8.2f} s")
print(f"  Pico de VRAM       : {vram_sem_cache:>8.1f} MB")
print(f"  Tokens por segundo : {NOVOS_TOKENS/t_sem_cache:>8.2f}")
print("-" * 52)
print("  Prévia do texto gerado:")
print("  " + textwrap.fill(texto_gerado_sc[:200], 50))
print("═" * 52)


## Otimização de Software: KV Cache

O KV Cache armazena as matrizes **K** e **V** calculadas no prefill.
Cada novo token só precisa calcular K/V para **si mesmo** e anexar ao cache:

```
SEM cache: recalcula 10.000 tokens   → O(n²) por passo  
COM cache: cache[10.000] + 1 token   → O(n)  por passo  
```

In [ ]:

print("[Passo 4a] Gerando COM KV Cache...")

modelo.config.use_cache = True
resetar_stats_vram()

t_inicio = time.perf_counter()

with torch.no_grad():
    saida_com_cache = modelo.generate(
        input_ids,
        max_new_tokens=NOVOS_TOKENS,
        do_sample=False,
        temperature=1.0,
        use_cache=True,
    )

t_com_cache    = time.perf_counter() - t_inicio
vram_com_cache = pico_vram_mb()

texto_gerado_cc = tokenizador.decode(
    saida_com_cache[0, n_tokens_entrada:], skip_special_tokens=True
)

acel_kv = t_sem_cache / t_com_cache if t_com_cache > 0 else float('inf')

print("═" * 52)
print("  MÉTRICA — Passo 4a: COM KV Cache")
print("═" * 52)
print(f"  Tempo de geração   : {t_com_cache:>8.2f} s")
print(f"  Pico de VRAM       : {vram_com_cache:>8.1f} MB")
print(f"  Tokens por segundo : {NOVOS_TOKENS/t_com_cache:>8.2f}")
print(f"   Aceleração      : {acel_kv:.2f}× mais rápido que sem cache!")
print("═" * 52)


##  Otimização de Hardware: FlashAttention-2

A atenção padrão escreve a matriz n×n completa na **HBM** (VRAM, lenta).
O FA2 fragmenta o cálculo em blocos que cabem na **SRAM** (on-chip, rápida):

```
Padrão : Q·Kᵀ → [n×n na HBM lenta] → softmax → ×V    OOM
FA2    : blocos na SRAM rápida → kernel fundido → saída   
```

 **T4 (Colab Free):** FA2 não é suportada → o bloco `try/except` usa fallback automático e o benchmark continua normalmente.

In [ ]:

FA2_DISPONIVEL = False
modelo_fa2     = None


del modelo
torch.cuda.empty_cache()
print(f"VRAM após liberar modelo anterior: {vram_livre_mb():.0f} MB livres")

try:
    print("\nCarregando modelo com FlashAttention-2...")
    resetar_stats_vram()

    modelo_fa2 = AutoModelForCausalLM.from_pretrained(
        ID_MODELO,
        quantization_config=config_bnb,
        device_map="auto",
        torch_dtype=torch.float16,
        attn_implementation="flash_attention_2",
        trust_remote_code=True,
    )
    modelo_fa2.config.use_cache = True
    modelo_fa2.eval()
    FA2_DISPONIVEL = True
    print(" FlashAttention-2 ativo!")

except (ImportError, RuntimeError, ValueError) as erro:
    print(f"  FA2 indisponível nesta GPU: {type(erro).__name__}")
    print("   Usando atenção padrão + KV Cache como fallback...")
    modelo_fa2 = AutoModelForCausalLM.from_pretrained(
        ID_MODELO,
        quantization_config=config_bnb,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    modelo_fa2.config.use_cache = True
    modelo_fa2.eval()

print(f"\nGerando {NOVOS_TOKENS} tokens...")
resetar_stats_vram()
t_inicio = time.perf_counter()

with torch.no_grad():
    saida_fa2 = modelo_fa2.generate(
        input_ids,
        max_new_tokens=NOVOS_TOKENS,
        do_sample=False,
        temperature=1.0,
        use_cache=True,
    )

t_fa2    = time.perf_counter() - t_inicio
vram_fa2 = pico_vram_mb()

texto_gerado_fa2 = tokenizador.decode(
    saida_fa2[0, n_tokens_entrada:], skip_special_tokens=True
)

acel_fa2 = t_sem_cache / t_fa2 if t_fa2 > 0 else float('inf')
rotulo   = "FlashAttention-2 + KV Cache" if FA2_DISPONIVEL else "Fallback + KV Cache"

print("═" * 52)
print(f"  MÉTRICA — Passo 4b: {rotulo}")
print("═" * 52)
print(f"  Tempo de geração   : {t_fa2:>8.2f} s")
print(f"  Pico de VRAM       : {vram_fa2:>8.1f} MB")
print(f"  Tokens por segundo : {NOVOS_TOKENS/t_fa2:>8.2f}")
print(f"   Aceleração total: {acel_fa2:.2f}× vs linha de base")
print("═" * 52)


##  Relatório Final e Gráficos

In [ ]:

print("\n" + "═" * 65)
print("  LAB 10 — RELATÓRIO FINAL DE BENCHMARK")
print("═" * 65)
print(f"  Modelo        : {ID_MODELO}")
print(f"  Quantização   : QLoRA 4-bit NF4 (dupla quantização)")
print(f"  Tokens entrada: {n_tokens_entrada:,}")
print(f"  Tokens gerados: {NOVOS_TOKENS}")
print(f"  FA2 ativo     : {' Sim' if FA2_DISPONIVEL else ' Fallback (T4 não suporta)'}")
print("─" * 65)
print(f"  {'Configuração':<35} {'Tempo':>6} {'TPS':>7} {'VRAM MB':>10}")
print("─" * 65)

linhas = [
    (" Sem KV Cache  (linha de base)", t_sem_cache,  vram_sem_cache),
    (" Com KV Cache  (sw)",            t_com_cache,  vram_com_cache),
    (" FA2 + KV Cache  (hw+sw)",      t_fa2,        vram_fa2),
]
for nome, t, vram in linhas:
    tps = NOVOS_TOKENS / t if t > 0 else 0
    print(f"  {nome:<35} {t:>5.1f}s {tps:>7.2f} {vram:>10.1f}")

print("─" * 65)
red_vram = ((vram_sem_cache - vram_fa2)/vram_sem_cache*100) if vram_sem_cache > 0 else 0
print(f"  Aceleração KV Cache            : {t_sem_cache/t_com_cache:.2f}×")
print(f"  Aceleração FA2 + KV Cache      : {t_sem_cache/t_fa2:.2f}×")
print(f"  Redução de pico de VRAM (FA2)  : {red_vram:.1f}%")
print("═" * 65)

In [ ]:

configs = [
    "Sem KV Cache\n(base)",
    "Com KV Cache\n(sw)",
    "FA2 + KV Cache\n(hw+sw)",
]
tempos = [t_sem_cache, t_com_cache, t_fa2]
vrms   = [vram_sem_cache, vram_com_cache, vram_fa2]
cores  = ["#e74c3c", "#f39c12", "#27ae60"]

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f"Lab 10 — Benchmark de Otimização GPU\n"
    f"({n_tokens_entrada:,} tokens entrada | {NOVOS_TOKENS} gerados | {ID_MODELO.split('/')[-1]})",
    fontsize=12, fontweight="bold"
)


b1 = eixos[0].bar(configs, tempos, color=cores, width=0.5, edgecolor="#2c3e50", linewidth=1.2)
eixos[0].set_title("⏱️ Tempo de Geração (menor = melhor)", fontsize=11)
eixos[0].set_ylabel("Segundos")
eixos[0].bar_label(b1, fmt="%.1f s", padding=4, fontsize=10)
eixos[0].set_ylim(0, max(tempos) * 1.3)
eixos[0].grid(axis="y", alpha=0.3)


b2 = eixos[1].bar(configs, vrms, color=cores, width=0.5, edgecolor="#2c3e50", linewidth=1.2)
eixos[1].set_title(" Pico de VRAM (menor = melhor)", fontsize=11)
eixos[1].set_ylabel("MB")
eixos[1].bar_label(b2, fmt="%.0f MB", padding=4, fontsize=10)
eixos[1].set_ylim(0, max(vrms) * 1.3 if max(vrms) > 0 else 500)
eixos[1].grid(axis="y", alpha=0.3)

legendas = [
    mpatches.Patch(color="#e74c3c", label="Linha de base (risco OOM)"),
    mpatches.Patch(color="#f39c12", label="+ KV Cache"),
    mpatches.Patch(color="#27ae60", label="+ FlashAttention-2"),
]
fig.legend(handles=legendas, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.08), fontsize=10)

plt.tight_layout()


try:
    from google.colab import drive
    plt.savefig("/content/lab10_benchmark.png", dpi=150, bbox_inches="tight")
    print(" Gráfico salvo em /content/lab10_benchmark.png")
except:
    plt.savefig("lab10_benchmark.png", dpi=150, bbox_inches="tight")
    print(" Gráfico salvo como lab10_benchmark.png")

plt.show()

---
## 📝 Análise Arquitetural (Passo 5 do README)

### Parte A — Como QLoRA + KV Cache + FlashAttention salvaram o Transformer

O problema central é uma **guerra de memória em três frentes**: pesos do modelo, ativações do prefill e estado da decodificação. Cada otimização resolve uma frente:

- **QLoRA 4-bit** → resolve o eixo dos *pesos*: de ~2.200 MB (FP16) para ~600 MB (NF4), tornando possível carregar o modelo na GPU.
- **KV Cache** → resolve o eixo da *decodificação*: de O(n²) por passo para O(n) por passo, reutilizando os tensores K e V do prefill.
- **FlashAttention-2** → resolve o eixo das *ativações no prefill*: processa a atenção em blocos na SRAM rápida, sem materializar a matriz n×n na HBM (VRAM lenta).

As três otimizações são **complementares e não redundantes** — juntas permitem que o pipeline de 15.000 tokens da HealthTech rode em uma única GPU.

### Parte B — Por que 2 milhões de tokens quebrariam até o FlashAttention, e por que Mamba é a solução

O FlashAttention-2 reduz o *coeficiente* na frente do O(n²), mas **não muda a complexidade quadrática**. Com 2 milhões de tokens, a matriz de atenção teria **4 × 10¹² elementos por cabeça** (~8 TB em FP16), e o próprio KV Cache ocuparia ~1 TB de VRAM. Nenhum hardware atual suporta isso.

A solução são os **Modelos de Espaço de Estados (SSMs)** como o **Mamba** (Gu & Dao, 2023): em vez de comparar todos os tokens entre si, o Mamba mantém um vetor de estado oculto de dimensão fixa, atualizado em **O(1) de memória por passo** e **O(n) de computação total**. Isso permite processar milhões de tokens com o mesmo orçamento de VRAM que um Transformer usa para milhares.

In [ ]:


salvar_no_drive = False

if salvar_no_drive:
    from google.colab import drive
    drive.mount('/content/drive')

    import os, shutil
    destino = '/content/drive/MyDrive/Lab10_Pipeline'
    os.makedirs(destino, exist_ok=True)

    shutil.copy('/content/lab10_benchmark.png', destino)
    print(f'Grafico salvo em: {destino}/lab10_benchmark.png')
    print()
    print('Para salvar o notebook no GitHub:')
    print('  Menu → Arquivo → Salvar uma cópia no GitHub')
else:
    print('Para salvar no Drive: mude salvar_no_drive = True e rode esta célula.')
    print()
    print('Para salvar o notebook direto no GitHub:')
    print('  Menu → Arquivo → Salvar uma cópia no GitHub')


In [ ]:

if modelo_fa2 is not None:
    del modelo_fa2
torch.cuda.empty_cache()

print(' Laboratório 10 finalizado!')
print()
print(' Para baixar o gráfico:')
print('   Painel lateral → Arquivos → lab10_benchmark.png → ícone de download')
print()
print(' Para salvar no GitHub direto do Colab:')
print('   Menu → Arquivo → Salvar uma cópia no GitHub')
print('   (autorize o Colab a acessar sua conta GitHub na janela que abre)')
print()
print('  Após subir, crie a tag v1.0 no terminal do seu PC:')
print('   git clone https://github.com/SEU_USUARIO/SEU_REPO')
print('   cd SEU_REPO')
print('   git tag -a v1.0 -m "Entrega final Lab10"')
print('   git push origin v1.0')
